### Setup

In [ ]:
import os

import torch
import torch.nn as nn
import wandb
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device]: {DEVICE}")

In [ ]:
mode = os.environ.get("CNN_REDUNDANCY_MODE", "max")
learning_rate = float(os.environ.get("CNN_REDUNDANCY_LR", "0.001"))
architecture_type = os.environ.get("CNN_REDUNDANCY_ARCH", "cnn")
epochs = int(os.environ.get("CNN_REDUNDANCY_EPOCHS", "10"))
batch_size = int(os.environ.get("CNN_REDUNDANCY_BATCH_SIZE", "64"))

config = {
    "mode": mode,
    "learning_rate": learning_rate,
    "architecture_type": architecture_type,
    "epochs": epochs,
    "batch_size": batch_size,
}

run = wandb.init(
    project="cnn-redundancy-reduction",
    config=config,
    name=f"{architecture_type}-{mode}",
)

MODE = run.config["mode"]
LEARNING_RATE = run.config["learning_rate"]
ARCHITECTURE_TYPE = run.config["architecture_type"]
EPOCHS = run.config["epochs"]
BATCH_SIZE = run.config["batch_size"]

### Dataset

In [ ]:
DATA_DIR    = "./data"
NUM_WORKERS = 2

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_ds = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("[train]:", len(train_ds), "[test]:", len(test_ds))
xb, yb = next(iter(train_loader))
print("[batch]:", tuple(xb.shape), tuple(yb.shape), xb.dtype)

### Architectures

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, in_ch=3, num_classes=10, width=64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(width)
        self.pool1 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(width, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(width * 2)
        self.conv4 = nn.Conv2d(width * 2, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn4   = nn.BatchNorm2d(width * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.conv5 = nn.Conv2d(width * 2, width * 4, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn5   = nn.BatchNorm2d(width * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.act     = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(width * 4, num_classes)

    def forward(self, x):
        convs = []

        c1, c1k = self.conv1(x), self.conv1.kernel_size
        convs.append(("cnn.conv1", c1, c1k))
        x = self.act(self.bn1(c1))

        c2, c2k = self.conv2(x), self.conv2.kernel_size
        convs.append(("cnn.conv2", c2, c2k))
        x = self.act(self.bn2(c2))
        x = self.pool1(x)

        c3, c3k = self.conv3(x), self.conv3.kernel_size
        convs.append(("cnn.conv3", c3, c3k))
        x = self.act(self.bn3(c3))

        c4, c4k = self.conv4(x), self.conv4.kernel_size
        convs.append(("cnn.conv4", c4, c4k))
        x = self.act(self.bn4(c4))
        x = self.pool2(x)

        c5, c5k = self.conv5(x), self.conv5.kernel_size
        convs.append(("cnn.conv5", c5, c5k))
        x = self.act(self.bn5(c5))
        x = self.pool3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        logits = self.fc(x)
        return logits, convs

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.skip_conv = None
        self.skip_bn   = None
        if stride != 1 or in_ch != out_ch:
            self.skip_conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0, bias=False)
            self.skip_bn   = nn.BatchNorm2d(out_ch)

        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        convs = []

        identity = x
        if self.skip_conv is not None and self.skip_bn is not None:
            ds, dsk = self.skip_conv(x), self.skip_conv.kernel_size
            convs.append(("downsample", ds, dsk))
            identity = self.skip_bn(ds)

        c1, c1k = self.conv1(x), self.conv1.kernel_size
        convs.append(("conv1", c1, c1k))
        out     = self.act(self.bn1(c1))

        c2, c2k = self.conv2(out), self.conv2.kernel_size
        convs.append(("conv2", c2, c2k))
        out     = self.bn2(c2)

        out = out + identity
        out = self.act(out)

        return out, convs


class ResNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=10, widths=(64, 128, 256, 512)):
        super().__init__()
        w1, w2, w3, w4 = widths

        self.stem_conv = nn.Conv2d(in_ch, w1, kernel_size=3, stride=1, padding=1, bias=False)
        self.stem_bn   = nn.BatchNorm2d(w1)
        self.stem_act  = nn.ReLU(inplace=True)

        self.layer1 = nn.ModuleList([ResBlock(w1, w1, 1), ResBlock(w1, w1, 1)])
        self.layer2 = nn.ModuleList([ResBlock(w1, w2, 2), ResBlock(w2, w2, 1)])
        self.layer3 = nn.ModuleList([ResBlock(w2, w3, 2), ResBlock(w3, w3, 1)])
        self.layer4 = nn.ModuleList([ResBlock(w3, w4, 2), ResBlock(w4, w4, 1)])

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(w4, num_classes)

    def forward(self, x):
        stem, stemk = self.stem_conv(x), self.stem_conv.kernel_size
        x = self.stem_act(self.stem_bn(stem))

        convs = []
        convs.append(("resnet.stem", stem, stemk))

        for i, block in enumerate(self.layer1):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer1.block{i}.{name}", c, k))
        for i, block in enumerate(self.layer2):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer2.block{i}.{name}", c, k))
        for i, block in enumerate(self.layer3):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer3.block{i}.{name}", c, k))
        for i, block in enumerate(self.layer4):
            x, bconvs = block(x)
            for name, c, k in bconvs:
                convs.append((f"resnet.layer4.block{i}.{name}", c, k))

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs


In [ ]:
class VGG16(nn.Module):
    def __init__(self, in_ch=3, num_classes=10):
        super().__init__()
        cfg = [
            64,
            64,
            "M",
            128,
            128,
            "M",
            256,
            256,
            256,
            "M",
            512,
            512,
            512,
            "M",
            512,
            512,
            512,
            "M",
        ]

        self.layers = nn.ModuleList()
        ch = in_ch
        for v in cfg:
            if v == "M":
                self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                out_ch = int(v)
                self.layers.append(nn.Conv2d(ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False))
                self.layers.append(nn.BatchNorm2d(out_ch))
                self.layers.append(nn.ReLU(inplace=True))
                ch = out_ch

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        convs = []

        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.Conv2d):
                c, ck = layer(x), layer.kernel_size
                x = c
                convs.append((f"vgg16.layers.{i}", c, ck))
            else:
                x = layer(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs


### Training